# 05 — Failure Analysis Demo
This notebook demonstrates how the system can analyze test failures using
artifacts (logs, traces, screenshots) and produce structured failure metadata.

The real failure-analysis engine lives in a private repository.
This notebook uses a simplified, public-safe simulation.


In [9]:
import json
from pathlib import Path
from datetime import datetime


## 1. Locate Latest Artifact Directory

We scan `artifacts-demo/` and load the most recent test run.


In [10]:
artifact_root = Path("../artifacts-demo")

# Find all test directories
test_dirs = [d for d in artifact_root.iterdir() if d.is_dir()]

# Find latest timestamped run
latest_run = None
latest_time = None

for test_dir in test_dirs:
    for run in test_dir.iterdir():
        if run.is_dir():
            try:
                ts = datetime.fromisoformat(run.name.replace("-", ":"))
                if latest_time is None or ts > latest_time:
                    latest_time = ts
                    latest_run = run
            except:
                pass

latest_run


## 2. Load Artifacts

We load:
- log.txt
- trace.json
- screenshot.png (not displayed, but path is used)


In [11]:
log_path = latest_run / "log.txt"
trace_path = latest_run / "trace.json"
screenshot_path = latest_run / "screenshot.png"

log_text = log_path.read_text()
trace_data = json.loads(trace_path.read_text())

log_text, trace_data, screenshot_path


TypeError: unsupported operand type(s) for /: 'NoneType' and 'str'

## 3. Mock Failure Classification Engine

In the real system:
- LLMs classify failures
- Heuristics detect common patterns
- Stack traces and logs are parsed
- UI diffs are analyzed

Here we simulate a simple rule-based classifier.


In [ ]:
def classify_failure(log_text, trace_data):
    if "timeout" in log_text.lower():
        return "timeout"
    if "selector" in log_text.lower():
        return "selector_not_found"
    if "assert" in log_text.lower():
        return "assertion_failure"
    return "unknown_failure"


## 4. Run Failure Classification


In [ ]:
failure_type = classify_failure(log_text, trace_data)
failure_type


## 5. Produce Structured Failure Metadata

This mirrors the structure used by the private engine.


In [ ]:
failure_metadata = {
    "test_name": latest_run.parent.name,
    "timestamp": latest_run.name,
    "failure_type": failure_type,
    "artifacts": {
        "log": str(log_path),
        "trace": str(trace_path),
        "screenshot": str(screenshot_path)
    },
    "analysis": {
        "summary": f"Detected failure type: {failure_type}",
        "suggested_next_steps": [
            "Review screenshot",
            "Inspect trace",
            "Check selector mappings",
            "Re-run test with debug mode"
        ]
    }
}

failure_metadata


## 6. Save Failure Report

We store the failure analysis in the same directory.


In [ ]:
report_path = latest_run / "failure_report.json"

with open(report_path, "w") as f:
    json.dump(failure_metadata, f, indent=4)

report_path


# Summary

In this notebook, we:

- Located the latest test run artifacts
- Loaded logs, traces, and screenshots
- Applied a mock failure classification engine
- Produced structured failure metadata
- Saved a failure report alongside artifacts

This completes the public demo pipeline:
Natural language → IR → Code → Execution → Artifacts → Failure Analysis
